### Oracle 서버와 파이썬 연동
- pip install oracledb
- pip install python-dotenv
- pip install faker

In [ ]:
import oracledb
from dotenv import load_dotenv      #python-dotenv 설치해서 가능한 것
import os

conn = oracledb.connect(user="python_user", password = "54321", dsn = "localhost:1521/FREEPDB1")
cursor = conn.cursor()

cursor.execute("select * from post")

for row in cursor.fetchall():
    print(row)

conn.close()

In [ ]:
# .env 파일 읽어오기
load_dotenv()

#  오라클 비밀번호 가져오기 (.env 가 없으면 기본값 사용)
password = os.getenv("ORACLE_PASSWORD", "54321")

In [ ]:
# 실습용 테이블 준비
# - category, member : 아래에서 사용하지만 python_user 스키마에 없던 테이블이라 여기서 생성
# - 노트북을 처음부터 다시 실행해도 PK 중복(ORA-00001)이 나지 않도록 기존 데이터는 비운다
# create table if not exists 는 Oracle 23ai 부터 지원
with oracledb.connect(user="python_user", password = password, dsn = "localhost:1521/FREEPDB1") as conn:
    with conn.cursor() as cursor:
        cursor.execute("""create table if not exists category(
                              category_id   number(10)   primary key,
                              category_name varchar2(50) not null
                          )""")
        cursor.execute("""create table if not exists member(
                              member_id number(10) primary key,
                              name      varchar2(50),
                              birth     date,
                              phone     varchar2(20),
                              joined_at date
                          )""")

        cursor.execute("delete from category")
        cursor.execute("delete from member")
        cursor.execute("delete from mart_member")
        conn.commit()

print("실습 테이블 준비 완료")

In [ ]:
# mart_member 는 age, rank, savings 도 NOT NULL 이라 같이 넣어야 ORA-01400 이 안 남
# sql = "insert into mart_member(member_id, name, password, age, rank, savings) values(:1, :2, :3, :4, :5, :6)"
sql = "insert into mart_member(member_id, name, password, age, rank, savings) values(:member_id, :name, :password, :age, :rank, :savings)"

with oracledb.connect(user="python_user", password = password, dsn = "localhost:1521/FREEPDB1") as conn:
    with conn.cursor() as cursor:
        cursor.execute(sql, ('hong123', '홍길동', 'hong12300', 30, 'silver', 1000))
        conn.commit()

In [ ]:
with oracledb.connect(user="python_user", password = password, dsn = "localhost:1521/FREEPDB1") as conn:
    with conn.cursor() as cursor:
        cursor.execute("select * from mart_member")
        for row in cursor.fetchall():
            print(row)

In [ ]:
sql = "insert into mart_member(member_id, name, password, age, rank, savings) values(:member_id, :name, :password, :age, :rank, :savings)"

with oracledb.connect(user="python_user", password = password, dsn = "localhost:1521/FREEPDB1") as conn:
    with conn.cursor() as cursor:
        cursor.execute(sql, ('kim123', '김길동', 'kim123000', 25, 'gold', 5000))
        conn.commit()

In [ ]:
# 딕셔너리(객체)로 줄 때 → 키 이름이 바인드 변수 이름과 정확히 일치해야 함
# values(:1, :2) 이렇게 주면 딕셔너리(객체)로 데이터를 선언못함. 

sql = "insert into mart_member(member_id, name, password, age, rank, savings) values(:member_id, :name, :password, :age, :rank, :savings)"

with oracledb.connect(user="python_user", password = password, dsn = "localhost:1521/FREEPDB1") as conn:
    with conn.cursor() as cursor:
        data = {
            "member_id" : "cho123", 
            "name" : "조미연", 
            "password" : "choi123456",
            "age" : 41,
            "rank" : "vip",
            "savings" : 12000
        }
        cursor.execute(sql, data)
        conn.commit()

In [ ]:
sql = "insert into category(category_id, category_name) values(:1, :2)"

# dsn 은 "호스트:포트/서비스명" -> localhost/xe 가 아니라 localhost:1521/FREEPDB1
with oracledb.connect(user="python_user", password = password, dsn = "localhost:1521/FREEPDB1") as conn:
    with conn.cursor() as cursor:
        cursor.execute(sql, (1101, '경제/경영'))
        conn.commit()

In [ ]:
with oracledb.connect(user="python_user", password = password, dsn = "localhost:1521/FREEPDB1") as conn:
    with conn.cursor() as cursor:
        cursor.execute("select * from category")
        for row in cursor.fetchall():
            print(row)

In [ ]:
sql = "insert into category(category_id, category_name) values(:1, :2)"

data = [
    [1102, '요리'],
    [1103, '예술'],
    [1104, '정치']
]

with oracledb.connect(user="python_user", password = password, dsn = "localhost:1521/FREEPDB1") as conn:
    with conn.cursor() as cursor:
        cursor.executemany(sql, data)
        conn.commit()

In [ ]:
# 사회 => 자기계발

sql = "update category set category_name=:1 where category_id = :2"

with oracledb.connect(user="python_user", password = password, dsn = "localhost:1521/FREEPDB1") as conn:
    with conn.cursor() as cursor:
        cursor.execute(sql, ("자기계발", 1104))
        conn.commit()

In [ ]:
# 삭제

sql = "delete from category where category_id = :1"

with oracledb.connect(user="python_user", password = password, dsn = "localhost:1521/FREEPDB1") as conn:
    with conn.cursor() as cursor:
        cursor.execute(sql, (1103,))
        conn.commit()

In [ ]:
from faker import Faker

fake = Faker(locale="ko_KR")
Faker.seed(4321)

print(fake.name())
print(fake.email())
print(fake.address())
print(fake.ssn())
print(fake.phone_number())

In [ ]:
from faker.providers import internet

fake.add_provider(internet)

for i in range(10):
    print(fake.ipv4_private())

In [ ]:
fake.date_of_birth(minimum_age=20, maximum_age=30)
fake.date_of_birth(minimum_age=20, maximum_age=30).strftime("%Y-%m-%d")

In [ ]:
fake.date_time_between(start_date="-3y", end_date="now")

In [ ]:
# mart_member에 50명의 임의의 데이터를 삽입하고 싶음
# name, birth, phone, joined_at

# 50명의 데이터를 리스트에 추가

faker = Faker(locale="ko_KR")
# Faker.seed(4321);

data = []

# i 값을 사용하지 않을때는 _ 로 사용하기도 함
for i in range(50):
    id = i
    name = faker.name()
    birth = faker.date_of_birth(minimum_age=19, maximum_age=70)   
    phone = faker.phone_number()
    joined_at = faker.date_time_between(start_date="-3y", end_date="now")  

    data.append([id, name, birth, phone, joined_at])

print(data)

sql = "insert into member(member_id, name, birth, phone, joined_at) values(:1, :2, :3, :4, :5)"

with oracledb.connect(user="python_user", password = password, dsn = "localhost:1521/FREEPDB1") as conn:
    with conn.cursor() as cursor:
        cursor.executemany(sql, data)
        conn.commit()
# insert로 실행